# 📦 VoxCity OBJ 내보내기

복셀 도시 모델과 시뮬레이션 결과를 외부 3D 소프트웨어에서 사용할 수 있도록 Wavefront OBJ 형식으로 내보냅니다.

## 내보내기 유형

| 내보내기 종류 | 함수 | 설명 |
|--------|----------|-------------|
| **복셀 도시** | `export_obj()` | 전체 3D 복셀 모델 |
| **시뮬레이션 결과** | `grid_to_obj()` | 값이 매핑된 색상화된 지표면 |

## 활용 사례

- **Blender/Rhino 연동** - 렌더링 및 추가 모델링을 위한 임포트
- **프레젠테이션** - 고품질 시각화 생성
- **분석 오버레이** - 시뮬레이션 결과를 색상화된 3D 지표면으로 확인

## 사전 요구 사항

```python
pip install voxcity
```

In [ ]:
# %pip install voxcity

from voxcity.generator import get_voxcity
from voxcity.exporter.obj import export_obj, grid_to_obj
from voxcity.simulator.solar import get_global_solar_irradiance_using_epw
from voxcity.simulator.view import get_view_index

meshsize = 5
rectangle_vertices = [
    (139.760, 35.680),
    (139.760, 35.690),
    (139.770, 35.690),
    (139.770, 35.680)
]

city = get_voxcity(
    rectangle_vertices,
    meshsize=meshsize,
    building_source='OpenStreetMap',
    land_cover_source='OpenStreetMap',
    canopy_height_source='High Resolution 1m Global Canopy Height Maps',
    dem_source='DeltaDTM',
    output_dir='output/obj_demo'
)

# VoxCity 객체에서 그리드 데이터에 접근
voxcity_grid = city.voxels.classes
dem_grid = city.dem.elevation

print(voxcity_grid.shape, dem_grid.shape)


---
## 🏙️ 복셀 도시 내보내기

전체 3D 복셀 도시 모델을 재질(Material) 정보가 포함된 OBJ 파일로 내보냅니다.

In [ ]:
export_obj(city, output_dir='output/obj_demo', file_name='voxcity')
print('복셀 도시 OBJ 내보내기 완료')


---
## 📊 시뮬레이션 결과를 색상화된 OBJ로 내보내기

2D 분석 그리드(일사량, 가시 지수 등)를 색상화된 3D 지표면으로 내보냅니다.

### `grid_to_obj()` 매개변수

| 매개변수 | 설명 |
|-----------|-------------|
| `output_dir` | 출력 디렉토리 경로 |
| `file_name` | 출력 파일 이름 (확장자 제외) |
| `cell_size` | 그리드 셀 크기 (meshsize) |
| `offset` | 지표면으로부터의 높이 오프셋 (기본값: 관찰자 시점 높이) |
| `colormap_name` | Matplotlib 컬러맵 이름 |
| `vmin`/`vmax` | 컬러 매핑을 위한 값 범위 |
| `alpha` | 투명도 (0 ~ 1) |

In [ ]:
# 순간 일사량
solar_kwargs = {
    "download_nearest_epw": True,
    "rectangle_vertices": rectangle_vertices,
    "calc_time": "01-01 12:00:00",
    "view_point_height": 1.5,
}
solar_grid = get_global_solar_irradiance_using_epw(
    city, calc_type='instantaneous', **solar_kwargs
)

# 순간 일사량을 색상화된 OBJ로 내보내기
grid_to_obj(
    solar_grid, dem_grid,
    output_dir='output/obj_demo', file_name='solar_instantaneous',
    cell_size=meshsize, offset=1.5, colormap_name='magma', num_colors=10, alpha=1.0,
    vmin=0
)

# 특정 기간 동안의 누적 일사량
cum_kwargs = solar_kwargs.copy()
cum_kwargs["start_time"] = "01-01 05:00:00"
cum_kwargs["end_time"] = "01-31 20:00:00"

cum_solar_grid = get_global_solar_irradiance_using_epw(
    city, calc_type='cumulative', **cum_kwargs
)

grid_to_obj(
    cum_solar_grid, dem_grid,
    output_dir='output/obj_demo', file_name='solar_cumulative',
    cell_size=meshsize, offset=1.5, colormap_name='viridis', num_colors=10, alpha=1.0
)

# 가시 지수
gvi = get_view_index(city, mode='green', obj_export=False)

grid_to_obj(
    gvi, dem_grid,
    output_dir='output/obj_demo', file_name='gvi',
    cell_size=meshsize, offset=1.5, colormap_name='Greens', num_colors=10, alpha=1.0,
    vmin=0.0, vmax=1.0
)
